**Instructions**
1.   Run the first script to install IfcOpenShell. This has to be done only once per session.
2.   Left side panel in the files tab upload 2 files: the excel file "IfcRaumimport_Vorlage.xlsx" with the spaces and the ifc-template "IfcRaumimport_Vorlage.ifc". If the files are called differently you have to change the name also in the script.
A warning will appear after upload, just click OK.
3.   Run the second script. It creates a new ifc file in the left panel under files that you can download. Filename: "Raumimport.ifc"
4.   Continue in AutoCAD:
5.   Import the IFC with the command IFCIMPORT  (tested with AutoCAD Architecture 2026)
6.   The IFC will be imported in another dwg and attached to your current drawing. Close your drawing and open the dwg where the rooms got placed. We do some cleanup:
7.   Select all rooms and detach all propertysets that you don't need
8.   Switch the roomstyle of all rooms to one of our styles, e.g. "Standard"
9.   Copy all the rooms to a brand new drawing to leave behind stuff that was imported by the ifc
10.   Place 1 roomtag by rightclicking on an existing tag > create similar > select one of the new rooms and Enter > OK > "M" for Multiple > Select all the new rooms > Enter > OK > Enter to finish
11.   All rooms are now tagged



In [4]:
# Installation of ifcopenshell
!pip install ifcopenshell

In [3]:
# Imports spaces from an excel file into an empty ifc template file. Script is set up to work with ifc2x3.
# An ifc-template is used so the script doesn't need to create the ifc file structure from scratch.
# Target area will be divided into the golden ratio.
# Excel file needs 5 columns: Abteilung, Raumnummer, Raumname, Anzahl, Sollfläche.
# Version: 0.3 - 27.04.2026

import ifcopenshell
import pandas as pd
import os

# Define a dictionary for excel columns so their names can be different in excel
column_map = {
    'Abteilung': 1,
    'Raumnummer': 2,
    'Raumname': 3,
    'Anzahl': 4,
    'Sollfläche': 5,
    'Info1': 6,
    'Info2': 7
}

# Read the Excel file into a DataFrame
df = pd.read_excel("IfcRaumimport_Vorlage.xlsx") # imports the first worksheet

# Resolve column indices to actual column names
resolved_columns = {
    key: df.columns[idx - 1]
    for key, idx in column_map.items()
}

# Loading the ifc file
input_filename = 'IfcRaumimport_Vorlage.ifc'
ifc_file = ifcopenshell.open(input_filename)

# Check for existing rooms and delete them
existing_spaces = ifc_file.by_type("IfcSpace")
if existing_spaces:
     print(f"Deleting {len(existing_spaces)} existing room.")
     for space in existing_spaces:
         ifc_file.remove(space)


# Get the existing context from the imported IFC file
context = ifc_file.by_type("IfcGeometricRepresentationContext")[0]

# Create a global placement for the site (at 0,0,0)
global_origin = ifc_file.createIfcCartesianPoint((0.0, 0.0, 0.0))
axis = ifc_file.createIfcDirection((0.0, 0.0, 1.0))
ref_direction = ifc_file.createIfcDirection((1.0, 0.0, 0.0))
axis2placement3d = ifc_file.createIfcAxis2Placement3D(global_origin, axis, ref_direction)
site_placement = ifc_file.createIfcLocalPlacement(None, axis2placement3d)

# Check if IfcSite exists, if not create one, and assign its placement
sites = ifc_file.by_type("IfcSite")
if not sites:
    site = ifc_file.createIfcSite(GlobalId=ifcopenshell.guid.new(), Name='Site', ObjectPlacement=site_placement)
elif len(sites) > 0:
    site = sites[0]
    # Ensure the existing site uses our defined global placement
    site.ObjectPlacement = site_placement

# Golden ratio, room height, and spacing parameters
golden_ratio = 1.618
room_height = 3.0
spacing = 1.0 # 1 meter gap between rooms

# Calculate total count of rooms (for printing output message)
total_count = df[resolved_columns['Anzahl']].sum()

# Initialize starting coordinates for the first room
x_coord = 10.0 # Start 10 meters in the X direction for the first column
y_coord = 0.0

# Initialize a counter for the space names, starting from 2
space_name_counter = 2

# List to hold all created spaces
all_created_spaces = []


# Iterate through the DataFrame rows to create multiple spaces
for index, row in df.iterrows():
    # Skip empty rows
    if row.isnull().all():
        continue

    # Get values from the current row using resolved column names
    abteilung = str(row[resolved_columns['Abteilung']]) if pd.notna(row[resolved_columns['Abteilung']]) else '-' # Get the Abteilung, replace with '-' if empty
    space_number = str(row[resolved_columns['Raumnummer']]) if pd.notna(row[resolved_columns['Raumnummer']]) else '-' # Convert to string, replace with '-' if empty
    space_name_long = str(row[resolved_columns['Raumname']]) if pd.notna(row[resolved_columns['Raumname']]) else '-' # Get the Raumname for LongName
    count = int(row[resolved_columns['Anzahl']])   # Convert to integer
    target_area = row[resolved_columns['Sollfläche']]
    info1 = str(row[resolved_columns['Info1']]) if 'Info1' in resolved_columns and pd.notna(row[resolved_columns['Info1']]) else None
    info2 = str(row[resolved_columns['Info2']]) if 'Info2' in resolved_columns and pd.notna(row[resolved_columns['Info2']]) else None

    # Check if Sollfläche is empty or zero and set to 1.0 if it is
    if pd.isna(target_area) or target_area == 0:
        target_area = 1.0

    # Creating a new property value Name1 (using Raumname for Name1 property)
    property_single_value_Name1 = ifc_file.create_entity(
        "IfcPropertySingleValue",
        "Name1",
        None,  # Description is optional
        ifc_file.create_entity("IfcText", space_name_long),  # Nominal value (using Raumname)
        None  # Unit is optional
    )

    # Creating a new property set wa_Raum
    property_set_Raum = ifc_file.create_entity(
        "IfcPropertySet",
        ifcopenshell.guid.new(),
        ifc_file.by_type("IfcOwnerHistory")[0],
        "wa_Raum",
        None,  # Description is optional
        [property_single_value_Name1]
    )

    # Creating a new property value Raumprogramm_Nummer
    property_single_value_RaumprogrammNummer = ifc_file.create_entity(
        "IfcPropertySingleValue",
        "Raumprogramm_Nummer",
        None,  # Description is optional
        ifc_file.create_entity("IfcText", space_number),  # Nominal value
        None  # Unit is optional
    )

    # Creating a new property value Raumprogramm_Sollfläche
    property_single_value_RaumprogrammSollfläche = ifc_file.create_entity(
        "IfcPropertySingleValue",
        "Raumprogramm_Sollfläche",
        None,  # Description is optional
        ifc_file.create_entity("IfcAreaMeasure", target_area),  # Nominal value
        None  # Unit is optional
    )

    # List to hold properties for wa_Raumprogramm
    raumprogramm_properties = [
        property_single_value_RaumprogrammNummer,
        property_single_value_RaumprogrammSollfläche
    ]

    # Conditionally add Raumprogramm_Info1
    if info1:
        property_single_value_RaumprogrammInfo1 = ifc_file.create_entity(
            "IfcPropertySingleValue",
            "Raumprogramm_Info1",
            None,
            ifc_file.create_entity("IfcText", info1),
            None
        )
        raumprogramm_properties.append(property_single_value_RaumprogrammInfo1)

    # Conditionally add Raumprogramm_Info2
    if info2:
        property_single_value_RaumprogrammInfo2 = ifc_file.create_entity(
            "IfcPropertySingleValue",
            "Raumprogramm_Info2",
            None,
            ifc_file.create_entity("IfcText", info2),
            None
        )
        raumprogramm_properties.append(property_single_value_RaumprogrammInfo2)

    # Creating a new property set wa_Raumprogramm
    property_set_Raumprogramm = ifc_file.create_entity(
        "IfcPropertySet",
        ifcopenshell.guid.new(),
        ifc_file.by_type("IfcOwnerHistory")[0],
        "wa_Raumprogramm",
        None,  # Description is optional
        raumprogramm_properties
    )


    # Inner loop to create duplicates based on count
    current_y_offset = y_coord
    for i in range(count):
        # Create IfcSpace entity (with unique GlobalId)
        # Use the counter for the Name and Raumname for the LongName
        ifc_space = ifc_file.createIfcSpace(
            GlobalId=ifcopenshell.guid.new(),
            Name=space_number, # Use Raumnummer as Name
            LongName=space_name_long, # Use Raumname as LongName
            CompositionType='ELEMENT' # Add CompositionType
        )

        # Increment the counter for the next space
        space_name_counter += 1

        # Calculate width and length based on golden ratio and target_area
        width = (target_area / golden_ratio) ** 0.5
        length = width * golden_ratio

        # Define current room's coordinates
        current_x = x_coord
        current_y = current_y_offset

        # Create geometry for the room
        point1 = ifc_file.createIfcCartesianPoint((0.0, 0.0, 0.0))
        point2 = ifc_file.createIfcCartesianPoint((length, 0.0, 0.0))
        point3 = ifc_file.createIfcCartesianPoint((length, width, 0.0))
        point4 = ifc_file.createIfcCartesianPoint((0.0, width, 0.0))


        polyline = ifc_file.createIfcPolyline([point1, point2, point3, point4, point1])
        profile = ifc_file.createIfcArbitraryClosedProfileDef(ProfileType='AREA', OuterCurve=polyline)
        extrusion = ifc_file.createIfcExtrudedAreaSolid(SweptArea=profile, Depth=room_height, ExtrudedDirection=ifc_file.createIfcDirection((0.0, 0.0, 1.0)))

        shape_representation = ifc_file.createIfcShapeRepresentation(ContextOfItems=context, RepresentationIdentifier='Body', RepresentationType='SweptSolid', Items=[extrusion])
        product_definition_shape = ifc_file.createIfcProductDefinitionShape(Representations=[shape_representation])

        # Create IfcLocalPlacement for the space
        object_placement = ifc_file.createIfcLocalPlacement(
            PlacementRelTo=site_placement,
            RelativePlacement=ifc_file.createIfcAxis2Placement3D(
                ifc_file.createIfcCartesianPoint((current_x, current_y, 0.0)), # Origin at the calculated coordinates
                ifc_file.createIfcDirection((0.0, 0.0, 1.0)), # Axis (Z-axis up)
                ifc_file.createIfcDirection((1.0, 0.0, 0.0))  # RefDirection (X-axis)
            )
        )

        # Assign the IfcLocalPlacement and Representation to the IfcSpace
        ifc_space.ObjectPlacement = object_placement
        ifc_space.Representation = product_definition_shape


        # Associate the space with property sets using IfcRelDefinesByProperties
        ifc_file.createIfcRelDefinesByProperties(
            GlobalId=ifcopenshell.guid.new(),
            RelatingPropertyDefinition=property_set_Raum,
            RelatedObjects=[ifc_space]
        )
        ifc_file.create_entity(
            "IfcRelDefinesByProperties",
            ifcopenshell.guid.new(),
            ifc_file.by_type("IfcOwnerHistory")[0],
            None,
            None,
            [ifc_space],
            property_set_Raumprogramm
        )


        # Add the created space to the list
        all_created_spaces.append(ifc_space)

        # Update y_offset for the next duplicate room
        current_y_offset += width + spacing

    # Update x_coord for the next set of rooms (from the next Excel row)
    # The length here refers to the length of the *last* room created in the inner loop for this row
    x_coord += length + spacing
    y_coord = 0.0 # Reset y_coord for the start of the next row of rooms

# Create a single IfcRelContainedInSpatialStructure for all spaces
if ifc_file.by_type("IfcOwnerHistory"):
    ifc_file.create_entity(
        "IfcRelContainedInSpatialStructure",
        ifcopenshell.guid.new(),
        ifc_file.by_type("IfcOwnerHistory")[0],
        None,
        None,
        all_created_spaces,
        ifc_file.by_guid(site.GlobalId)
    )
else:
     # Handle the case where there is no OwnerHistory
     ifc_file.create_entity(
        "IfcRelContainedInSpatialStructure",
        ifcopenshell.guid.new(),
        None,
        None,
        None,
        all_created_spaces,
        ifc_file.by_guid(site.GlobalId)
    )



# Construct the output filename
base_name, ext = os.path.splitext(input_filename)
output_filename = f"Raumimport{ext}"

# Save the modified IFC file
ifc_file.write(output_filename)
print(f"IFC created: {output_filename} with {int(total_count)} rooms")

Deleting 1 existing room.
IFC created: Raumimport.ifc with 22 rooms
